In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## For experimentation Tracking

In [6]:
!pip install mlflow dagshub -qqq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 112.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.1/260.1 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.9/139.9 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 700.0/700.0 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.2/203.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [7]:

import os

os.environ["MLFLOW_TRACKING_USERNAME"] = "iambikash378"
os.environ["MLFLOW_TRACKING_PASSWORD"] = ""


In [8]:
import dagshub
dagshub.init(repo_owner='iambikash378', repo_name='otitis_media', mlflow=True)

mlflow.set_tracking_uri(f"https://dagshub.com/iambikash378/otitis_media.mlflow/")


❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=fe9c9320-0358-4d23-b1eb-662304e40487&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=35c273fab2294a49010917542e8eb7c6b4f04cf3c25ca2e152e7be2fdec128d4




KeyboardInterrupt: 

## Dataset Exploration

In [9]:
training_dataset_path = '/content/drive/MyDrive/Datasets/Datos/Training-validation'

In [10]:
os.getcwd()

'/content'

### Total Samples

In [11]:
def count_files(dataset_root_folder):
    total = 0
    for dirpath, dirnames, filenames in os.walk(dataset_root_folder):
        if dirpath != dataset_root_folder:
          dirname = os.path.basename(dirpath)
          numfiles = len(filenames)
          print(f'{dirname} : {numfiles}')
        total += len(filenames)
    print(f"Total number of images : {total}")


In [12]:
count_files(training_dataset_path)

Normal : 180
Chronic otitis media : 180
Earwax plug : 180
Myringosclerosis : 180
Total number of images : 720


In [13]:
from PIL import Image
import numpy as np


img = Image.open('/content/drive/MyDrive/Datasets/Datos/Training-validation/Normal/n101.jpg')

img_array = np.array(img)

print(f"data type : {img_array.dtype}")

min_val = img_array.min()
max_val = img_array.max()

print(f"Min- Max pixel value: {min_val} and {max_val}")

data type : uint8
Min- Max pixel value: 8 and 255


In [14]:

# def assert_same_image_size(root_dir):
#     expected_size = None

#     for dirpath, _, filenames in os.walk(root_dir):
#         for filename in filenames:
#             if filename.lower().endswith(('.jpg', '.jpeg')):
#                 file_path = os.path.join(dirpath, filename)
#                 with Image.open(file_path) as img:
#                     size = img.size

#                     if expected_size is None:
#                         expected_size = size
#                         print(f"Reference size: {expected_size}")
#                     else:
#                         if size != expected_size:
#                             raise ValueError(f"Image {file_path} has size {size}, expected {expected_size}")

#     print(" All images have the same size:", expected_size)

# assert_same_image_size('/content/drive/MyDrive/Datasets/Datos')


In [15]:
from torchvision import transforms

IMG_SIZE = 256

img_transform = transforms.Compose([ transforms.Resize((IMG_SIZE, IMG_SIZE)),
                                transforms.ToTensor(),
                                 transforms.Normalize(
                                            (0.485, 0.456, 0.406),
                                            (0.229, 0.224, 0.225)
                                 )
])

In [16]:
num_classes = 3 #Normal vs Abnormal
normal_class = ['Normal']
abnormal_class = ['Myringosclerosis', 'Chronic otitis media']
earwax_class = ['Earwax plug']

In [17]:
# for eachdir in os.listdir('/content/drive/MyDrive/Datasets/Datos/Training-validation'):
#   if eachdir in normal_class:


In [18]:
val_ratio = 0.2

In [19]:
import csv
def build_csv(img_path):
  pass


## Dataset Loader

In [20]:
from torch.utils.data import Dataset

In [21]:
class datos_dataset(Dataset):
  def __init__(self, img_dir, transform = None):
    self.img_paths = []
    self.labels = []
    self.class_to_label = {}
    self.transform = transform

    for eachfolder in os.listdir(img_dir):
      class_dir_path = os.path.join(img_dir, eachfolder)

      if eachfolder in normal_class:
        label = 0

      elif eachfolder in abnormal_class:
        label = 1

      elif eachfolder in earwax_class:
        label = 2

      if os.path.isdir(class_dir_path):
            self.class_to_label[eachfolder] = label

      for filename in os.listdir(class_dir_path):
        if filename.lower().endswith(('.jpg', '.jpeg')):
          self.img_paths.append(os.path.join(class_dir_path, filename))
          self.labels.append(label)


  def __len__(self):
    return len(self.img_paths)

  def __getitem__(self, index):
    img = self.img_paths[index]
    label = self.labels[index]

    image = Image.open(img).convert("RGB")

    if self.transform:
      image = self.transform(image)

    return image, label


In [22]:
train_data = datos_dataset('/content/drive/MyDrive/Datasets/Datos/Training-validation', transform = img_transform)

In [23]:
print(train_data.__len__())
print(train_data.__getitem__(500))

720
(tensor([[[-1.5528, -1.5528, -1.5528,  ..., -1.4843, -1.4672, -1.4672],
         [-1.5528, -1.5528, -1.5528,  ..., -1.4843, -1.4672, -1.4672],
         [-1.5528, -1.5528, -1.5528,  ..., -1.4843, -1.4672, -1.4672],
         ...,
         [-1.5357, -1.5357, -1.5528,  ..., -1.5014, -1.5014, -1.5014],
         [-1.5357, -1.5357, -1.5528,  ..., -1.5014, -1.5014, -1.5014],
         [-1.5357, -1.5357, -1.5528,  ..., -1.4843, -1.5014, -1.5014]],

        [[-1.3880, -1.3880, -1.3880,  ..., -1.3704, -1.3880, -1.3880],
         [-1.4055, -1.4055, -1.4055,  ..., -1.3704, -1.3880, -1.3880],
         [-1.4055, -1.4055, -1.4055,  ..., -1.3704, -1.3880, -1.3880],
         ...,
         [-1.4055, -1.4055, -1.4230,  ..., -1.4755, -1.4580, -1.4580],
         [-1.4055, -1.4055, -1.4230,  ..., -1.4755, -1.4580, -1.4580],
         [-1.4055, -1.4055, -1.4230,  ..., -1.4580, -1.4580, -1.4580]],

        [[-0.9504, -0.9504, -0.9504,  ..., -0.9156, -0.9156, -0.9156],
         [-0.9330, -0.9330, -0.9330,  ..

In [24]:
print(train_data.class_to_label)

{'Normal': 0, 'Chronic otitis media': 1, 'Earwax plug': 2, 'Myringosclerosis': 1}


In [25]:
train_size = int((1-val_ratio)*len(train_data))
val_size = len(train_data) - train_size

In [26]:
from torch.utils.data import DataLoader, random_split

train_dataset, val_dataset = random_split(train_data, [train_size, val_size])
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [27]:
dataset_path = '/content/drive/MyDrive/Datasets/Datos'

In [28]:
def cross_entropy_loss():
  pass


## Training Loop

In [29]:
EPOCHS = 200
PATIENCE = 20
LEARNING_RATE = 1e-4
KERNEL_SIZE = 3
BATCH_SIZE = 32

In [30]:
from torch.nn import CrossEntropyLoss

## Pretrained Resnet18 Model

In [31]:
from sklearn.metrics import precision_score, recall_score, f1_score

In [32]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"

In [33]:
import torchvision.models as models
model = models.resnet18(pretrained=True)
print(model.__class__.__name__)
print(next(model.parameters()).device)

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 98.1MB/s]


ResNet
cpu


In [34]:
import torch
loss_fn = CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = None

print(optimizer)


Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0001
    maximize: False
    weight_decay: 0
)


In [35]:
import torch
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)
model.to(device)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [36]:
best_val_loss = float('inf')
remarks = "first"
MODEL_SAVE_PATH = f'/content/drive/MyDrive/Trained_Models/{remarks}.pth'
counter = 0

In [37]:
with mlflow.start_run():
  mlflow.log_params(
    {
        "no. of classes" : num_classes,
        "classes" : train_data.class_to_label,
        "initial_learning_rate": LEARNING_RATE,
        "batch_size": BATCH_SIZE,
        "model_name": model.__class__.__name__,
        "model_arch": model,
        "loss_function": loss_fn,
        "epochs": EPOCHS,
        "patience": PATIENCE,
        "optimizer": optimizer,
        "scheduler" : scheduler if scheduler is not None else None ,
        "validation split" : val_ratio,
        "image size" : IMG_SIZE,
        "saved model remarks" : remarks
    }
    )
  for epoch in range(EPOCHS):
    model.train()
    train_running_loss = 0.0
    train_running_corrects = 0

    for inputs, labels in train_loader:
      inputs = inputs.to(device)
      labels = labels.to(device)

      optimizer.zero_grad()

      output = model(inputs)

      _, preds = torch.max(output, 1)

      loss = loss_fn(output, labels)

      loss.backward()

      optimizer.step()

      train_running_loss += loss.item() * inputs.size(0)
      train_running_corrects += torch.sum(preds == labels)

    train_loss = train_running_loss / len(train_loader.dataset)
    train_accuracy = train_running_corrects / len(train_loader.dataset)

    model.eval()

    val_running_loss = 0.0
    val_running_corrects = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():

      for inputs, labels in val_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        output = model(inputs)

        _, preds = torch.max(output, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

        loss = loss_fn(output, labels)

        val_running_loss += loss.item() * inputs.size(0)

        val_running_corrects += torch.sum(preds == labels)

      val_loss = val_running_loss / len(val_loader.dataset)
      val_accuracy = val_running_corrects / len(val_loader.dataset)

      all_preds = np.array(all_preds)
      all_labels = np.array(all_labels)

      precision = precision_score(all_labels, all_preds, average=None, zero_division = 0)
      recall = recall_score(all_labels, all_preds, average=None, zero_division = 0)
      f1 = f1_score(all_labels, all_preds, average=None, zero_division = 0)

      precision_normal, recall_normal, f1_normal = precision[0], recall[0], f1[0]
      precision_abnormal, recall_abnormal, f1_abnormal = precision[1], recall[1], f1[1]
      precision_earwax , recall_earwax, f1_earwax = precision[2], recall[2], f1[2]

      mlflow.log_metric("val_loss", val_loss, step=epoch)
      mlflow.log_metric("val_accuracy", val_accuracy, step=epoch)

      mlflow.log_metric("Normal Precision", precision_normal, step=epoch)
      mlflow.log_metric("Normal Recall", recall_normal, step=epoch)
      mlflow.log_metric("Normal F1", f1_normal, step=epoch)

      mlflow.log_metric("Abnormal Precision", precision_abnormal, step=epoch)
      mlflow.log_metric("Abnormal Recall", recall_abnormal, step=epoch)
      mlflow.log_metric("Abnormal F1", f1_abnormal, step=epoch)

      mlflow.log_metric("Earwax Precision", precision_earwax, step=epoch)
      mlflow.log_metric("Earwax Recall", recall_earwax, step=epoch)
      mlflow.log_metric("Earwax F1", f1_earwax, step=epoch)

      print(f'Epoch [{epoch+1}], train loss: {train_loss:.4f}, train acc: {train_accuracy:.4f}, val loss: {val_loss:.4f}, val acc: {val_accuracy:.4f}')

      if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        mlflow.log_artifact(MODEL_SAVE_PATH)
        print(f"Saved best model with val loss : {val_loss:.4f}")
      else:
        counter += 1
        print(f"No improvement in validation loss for {counter} epochs.")

      if counter >= PATIENCE:
        print(f"Early stopping triggered after {PATIENCE} epochs without improvement.")
        break

      #scheduler.step(val_loss)



NameError: name 'mlflow' is not defined

In [41]:
# get a single batch from trainloader

images, labels = next(iter(train_loader))
output = model(images)
print(output)

print(torch.max(output,1))

values, indices = torch.max(output,1)
total_correct = torch.sum(indices == labels)
accuracy = total_correct / len(labels)
print(f"Accuracy: {accuracy}")

tensor([[-1.2006e+00, -4.5316e-01, -5.0898e-02],
        [-1.1669e+00,  4.5349e-01, -2.3461e-01],
        [-1.0457e+00, -3.4881e-01,  4.9628e-01],
        [-2.2977e-01, -4.1531e-01, -9.6087e-02],
        [-1.3085e+00,  3.8288e-01,  1.2294e-01],
        [-1.4006e+00,  4.8517e-01, -2.0549e-01],
        [-1.2641e+00,  1.0070e+00, -1.0487e+00],
        [-7.6391e-01,  2.3836e-01, -1.8159e-01],
        [-1.4705e+00,  2.7574e-01, -5.6395e-01],
        [-2.0444e-01,  8.2749e-02, -3.4140e-01],
        [-8.4875e-01, -3.6841e-01, -5.1478e-01],
        [-3.5304e-01,  2.5028e-01, -3.9823e-01],
        [-1.1416e+00,  6.8428e-01, -6.6161e-02],
        [-4.0280e-01, -1.4579e-01,  7.4127e-01],
        [-9.9268e-01, -9.6406e-02, -1.6049e-01],
        [-6.2044e-01, -3.7160e-01,  4.4689e-01],
        [-9.4241e-01,  2.9141e-01,  1.6194e-01],
        [-1.7670e+00,  6.5161e-01, -6.3178e-02],
        [-1.2880e+00,  6.2378e-02, -5.5043e-01],
        [-1.0866e+00, -6.4110e-01, -5.0470e-01],
        [-1.1151e+00

## Evaluation Loop

In [ ]:
testing_dataset_path = '/content/drive/MyDrive/Datasets/Datos/Testing'
test_data = datos_dataset(testing_dataset_path, transform = img_transform)
test_dataloader = DataLoader(test_data, batch_size = 32, shuffle = False)

In [ ]:
model.eval()

model.load_state_dict(torch.load('/content/drive/MyDrive/Trained_Models/first.pth', map_location = device))
model = model.to(device)

val_running_loss = 0.0
val_running_corrects = 0

all_preds = []
all_labels = []


with torch.no_grad():

  with mlflow.start_run():
    mlflow.log_params(
      {
          "no. of classes" : num_classes,
          "classes" : train_data.class_to_label,
          "initial_learning_rate": LEARNING_RATE,
          "batch_size": BATCH_SIZE,
          "model_name": model.__class__.__name__,
          "model_arch": model,
          "loss_function": loss_fn,
          "epochs": EPOCHS,
          "patience": PATIENCE,
          "optimizer": optimizer,
          "scheduler" : scheduler if scheduler is not None else None ,
          "validation split" : val_ratio,
          "image size" : IMG_SIZE,
          "saved model remarks" : remarks
      }
      )

    for inputs, labels in test_dataloader:
      inputs = inputs.to(device)
      labels = labels.to(device)

      output = model(inputs)

      _, preds = torch.max(output, 1)

      all_preds.extend(preds.cpu().numpy())
      all_labels.extend(labels.cpu().numpy())

      loss = loss_fn(output, labels)

      val_running_loss += loss.item() * inputs.size(0)

      val_running_corrects += torch.sum(preds == labels)

    val_loss = val_running_loss / len(val_loader.dataset)
    val_accuracy = val_running_corrects / len(val_loader.dataset)

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    precision = precision_score(all_labels, all_preds, average=None, zero_division = 0)
    recall = recall_score(all_labels, all_preds, average=None, zero_division = 0)
    f1 = f1_score(all_labels, all_preds, average=None, zero_division = 0)

    precision_normal, recall_normal, f1_normal = precision[0], recall[0], f1[0]
    precision_abnormal, recall_abnormal, f1_abnormal = precision[1], recall[1], f1[1]
    precision_earwax , recall_earwax, f1_earwax = precision[2], recall[2], f1[2]

    mlflow.log_metric("val_loss", val_loss, step=epoch)
    mlflow.log_metric("val_accuracy", val_accuracy, step=epoch)

    mlflow.log_metric("Normal Precision", precision_normal, step=epoch)
    mlflow.log_metric("Normal Recall", recall_normal, step=epoch)
    mlflow.log_metric("Normal F1", f1_normal, step=epoch)

    mlflow.log_metric("Abnormal Precision", precision_abnormal, step=epoch)
    mlflow.log_metric("Abnormal Recall", recall_abnormal, step=epoch)
    mlflow.log_metric("Abnormal F1", f1_abnormal, step=epoch)

    mlflow.log_metric("Earwax Precision", precision_earwax, step=epoch)
    mlflow.log_metric("Earwax Recall", recall_earwax, step=epoch)
    mlflow.log_metric("Earwax F1", f1_earwax, step=epoch)


## Testing Individual Images